In [ ]:
import os; os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [1]:
import torch; torch.cuda.empty_cache()
!nvidia-smi

Mon Mar 30 22:45:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:A4:00.0 Off |                    0 |
| N/A   29C    P0             69W /  700W |       1MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
import json

training_data = []
with open('data/train.jsonl', 'r') as f:
    for line in f.readlines():
        if line.strip():
            data = json.loads(line)
            training_data.append(data)

print(training_data[0])


{'expression': '25 - 14', 'answer': 11, 'type': 'standard'}


In [4]:
from datasets import Dataset
dataset = Dataset.from_list(training_data)
dataset

Dataset({
    features: ['expression', 'answer', 'type'],
    num_rows: 1600
})

In [5]:
SYSTEM_PROMPT = (
    "Evaluate the arithmetic expression using the symbol definitions provided. "
    "There are 4 symbols θ, α, γ, β each representing one of the four basic arithmetic operations (+, -, ×, ÷). "
    "Each symbol maps to exactly one operation. Standard operator precedence (BODMAS) applies. "
    "Be concise. Put your final answer in \\boxed{}."
)

SYSTEM_PROMPT

'Evaluate the arithmetic expression using the symbol definitions provided. There are 4 symbols θ, α, γ, β each representing one of the four basic arithmetic operations (+, -, ×, ÷). Each symbol maps to exactly one operation. Standard operator precedence (BODMAS) applies. Be concise. Put your final answer in \\boxed{}.'

In [6]:
dataset = dataset.map(lambda x: {
    "prompt": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Expression: {x['expression']} ->/no_think"}
    ]
})
dataset[0]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

{'expression': '25 - 14',
 'answer': 11,
 'type': 'standard',
 'prompt': [{'content': 'Evaluate the arithmetic expression using the symbol definitions provided. There are 4 symbols θ, α, γ, β each representing one of the four basic arithmetic operations (+, -, ×, ÷). Each symbol maps to exactly one operation. Standard operator precedence (BODMAS) applies. Be concise. Put your final answer in \\boxed{}.',
   'role': 'system'},
  {'content': 'Expression: 25 - 14 ->/no_think', 'role': 'user'}]}

In [7]:
dataset.shuffle(seed=23)

Dataset({
    features: ['expression', 'answer', 'type', 'prompt'],
    num_rows: 1600
})

In [8]:
import dotenv
dotenv.load_dotenv(override=True)

import os
wandb_project = os.getenv("WANDB_PROJECT")
wandb_entity = os.getenv("WANDB_ENTITY")
wandb_run_name = "test-v1-no-few-shot-examples-adapter-only"

wandb_project, wandb_entity, wandb_run_name

('api-adapter', 'ronny21', 'test-v1-no-few-shot-examples-adapter-only')

In [9]:
from pathlib import Path
from trl.trainer import GRPOConfig

output_dir = Path("outputs/grpo-adapter-only")
max_steps = 1000
per_device_train_batch_size = 16
per_device_eval_batch_size = 64
gradient_accumulation_steps = 1
learning_rate = 5e-6
num_generations = 64

config = GRPOConfig(
    output_dir=str(output_dir),
    run_name=wandb_run_name,
    max_steps=max_steps,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=learning_rate,
    weight_decay=0.001,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    num_generations=num_generations,
    generation_batch_size=num_generations,
    max_completion_length=512,
    temperature=1.0,
    top_k=50,
    logging_steps=50,
    save_steps=200,
    bf16=True,
    report_to="wandb",
    log_completions=True,
    num_completions_to_print=5,
    eval_strategy="steps",
    eval_steps=100,
    do_eval=True,
)
config

UnslothGRPOConfig(output_dir='outputs/grpo-adapter-only', overwrite_output_dir=None, do_train=False, do_eval=True, do_predict=False, eval_strategy=<IntervalStrategy.STEPS: 'steps'>, prediction_loss_only=False, per_device_train_batch_size=16, per_device_eval_batch_size=64, per_gpu_train_batch_size=None, per_gpu_eval_batch_size=None, gradient_accumulation_steps=1, eval_accumulation_steps=2, eval_delay=0, torch_empty_cache_steps=250, learning_rate=5e-06, weight_decay=0.001, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, max_grad_norm=1.0, num_train_epochs=3.0, max_steps=1000, lr_scheduler_type=<SchedulerType.LINEAR: 'linear'>, lr_scheduler_kwargs=None, warmup_ratio=0.1, warmup_steps=0, log_level='passive', log_level_replica='warning', log_on_each_node=True, logging_dir='outputs/grpo-adapter-only/runs/Mar30_22-45-16_ai-innovation-h100-02-preserve', logging_strategy=<IntervalStrategy.STEPS: 'steps'>, logging_first_step=False, logging_steps=50, logging_nan_inf_filter=False, save_strat

In [10]:
import re
ptrn = re.compile(r"\\boxed\{(.*)\}")

a = ptrn.search("1 + 2 = \\boxed{3}")
print(a.group(1))

b = ptrn.search("1 + 2 = 3")
print(b)

3
None


In [11]:
def correctness_reward_fn(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]["content"] for completion in completions]
    # extract the boxed answer from each response
    rewards = []
    for response, ans in zip(responses, answer):
        try:
            _ans = ptrn.search(response)
            rewards.append(float(float(_ans.group(1)) == float(ans)))
        except Exception as e:
            rewards.append(0.0)
        # print(f"response: {response}, true_ans: {ans}, reward: {rewards[-1]}")
    return rewards
    



In [12]:
# test correctness_reward_fn
prompts = ["1 + 2 = \\boxed{3}", "1 + 2 = 3"]
completions = [[{'content':"1 + 2 = \\boxed{3}"}], [{'content':"1 + 2 = 4"}]]
answer = ["3", "4"]
correctness_reward_fn(prompts, completions, answer)


[1.0, 0.0]

In [13]:

DEFAULT_MODEL_NAME = "unsloth/Qwen3-8B"
DEFAULT_MAX_SEQ_LENGTH = 2048
DEFAULT_LORA_RANK = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=DEFAULT_MODEL_NAME,
    max_seq_length=DEFAULT_MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect (bf16 on H100)
    load_in_4bit=False,
    fast_inference=True,
    max_lora_rank=DEFAULT_LORA_RANK,
    gpu_memory_utilization=0.5,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=DEFAULT_LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=DEFAULT_LORA_RANK * 2,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

FastLanguageModel.for_training(model)

INFO 03-30 22:45:26 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.17.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 8. Max memory: 79.097 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/Qwen3-8B with actual GPU utilization = 49.51%
Unsloth: Your GPU has CUDA compute capability 9.0 with VRAM = 79.1 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 80.
Unsloth: vLLM's KV Cache can use up to 23.59 GB. Also swap space = 6 GB.
Unsloth: Not an error, but `use_cudagraph` is not supported in vLLM.config.CompilationConfig. Skipping.
Unsloth: Not an error, but `u

/workspace/home/lab/rawhad/api-adapter/.venv/lib/python3.10/site-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


WARNING 03-30 22:45:39 [arg_utils.py:1321] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 03-30 22:45:41 [model.py:531] Resolved architecture: Qwen3ForCausalLM
INFO 03-30 22:45:41 [model.py:1554] Using max model len 2048
INFO 03-30 22:45:41 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=6144.
INFO 03-30 22:45:41 [vllm.py:747] Asynchronous scheduling is enabled.
INFO 03-30 22:45:43 [core.py:101] Initializing a V1 LLM engine (v0.17.0) with config: model='unsloth/Qwen3-8B', speculative_config=None, tokenizer='unsloth/Qwen3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enfor

/workspace/home/lab/rawhad/api-adapter/.venv/lib/python3.10/site-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


INFO 03-30 22:45:44 [topk_topp_sampler.py:51] Using FlashInfer for top-p & top-k sampling.
INFO 03-30 22:45:44 [base.py:106] Offloader set to NoopOffloader
INFO 03-30 22:45:44 [gpu_model_runner.py:4255] Starting to load model unsloth/Qwen3-8B...
INFO 03-30 22:45:44 [cuda.py:405] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION'].
INFO 03-30 22:45:44 [flash_attn.py:587] Using FlashAttention version 3


...

INFO 03-30 22:46:05 [core.py:282] init engine (profile, create kv cache, warmup model) took 15.22 seconds
INFO 03-30 22:46:07 [llm.py:388] Supported tasks: ('generate',)
Unsloth: Just some info: will skip parsing ['pre_feedforward_layernorm', 'ffn_norm', 'norm1', 'layer_norm1', 'layer_norm2', 'input_layernorm', 'norm', 'post_layernorm', 'post_attention_layernorm', 'norm2', 'k_norm', 'q_norm', 'attention_norm', 'post_feedforward_layernorm']


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['pre_feedforward_layernorm', 'ffn_norm', 'norm1', 'cross_attn_post_attention_layernorm', 'layer_norm1', 'layer_norm2', 'input_layernorm', 'norm', 'cross_attn_input_layernorm', 'post_layernorm', 'post_attention_layernorm', 'norm2', 'k_norm', 'q_norm', 'attention_norm', 'post_feedforward_layernorm']
unsloth/Qwen3-8B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 4096, padding_idx=151669)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [14]:
train_test_split = dataset.train_test_split(test_size=0.2)
train_dataset = train_test_split["train"]
eval_dataset = train_test_split["test"]

train_dataset, eval_dataset

(Dataset({
     features: ['expression', 'answer', 'type', 'prompt'],
     num_rows: 1280
 }),
 Dataset({
     features: ['expression', 'answer', 'type', 'prompt'],
     num_rows: 320
 }))

In [15]:
from trl.trainer import GRPOTrainer

# Workaround: TRL expects warnings_issued on the model but PEFT doesn't expose it
if not hasattr(model, "warnings_issued"):
    model.warnings_issued = {}

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[correctness_reward_fn],
    args=config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,280 | Num Epochs = 1 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 87,293,952 of 8,278,029,312 (1.05% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


WARNING 03-30 22:46:19 [input_processor.py:168] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
